Data Formatting

In [1]:
import json

# Read the JSONL file and update the dietary_info field
updated_lines = []
with open("products_info.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        product = json.loads(line.strip())

        # Add "Non-Vegetarian" if dietary_info is empty or missing
        if not product["dietary_info"]:
            product["dietary_info"] = ["Non-Vegetarian"]

        updated_lines.append(json.dumps(product))

# Write the updated data back to the JSONL file
with open("products_info.jsonl", "w", encoding="utf-8") as file:
    file.write("\n".join(updated_lines))

print("Updated dietary_info in products_info.jsonl")


Updated dietary_info in products_info.jsonl


In [ ]:
import json

# List to store formatted strings
text_data = []

# Read the JSONL file and process each line
with open("products_info.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        product = json.loads(line.strip())  # Convert JSONL line to dictionary
        
        # Convert each key-value pair to a formatted string
        text_representation = []
        for key, value in product.items():
            if isinstance(value, list):  # Convert lists to comma-separated strings
                value = ", ".join(map(str, value))
            elif isinstance(value, dict):  # Convert dicts to key-value pairs
                value = ", ".join(f"{k}: {v}" for k, v in value.items())
            text_representation.append(f"{key}: {value}")  # Append formatted key-value pair
        
        text_data.append(", ".join(text_representation))  # Merge into a single string


In [8]:
with open(r"about_cs/kafi_about_us.txt", "r", encoding="utf-8") as file:
    about_text = file.read().strip()  # Read and remove any extra spaces or newlines

text_data.append(about_text)  # Append to the existing text_data list


In [9]:
text_data[-1]

"Welcome to Kafi, your neighborhood coffee shop in the heart of Tadipatri, Andhra Pradesh. At Kafi, we believe coffee is more than just a drink—it’s an experience, a moment of joy, and a way to connect with others.\n\nOur Story\nFounded in 2015 by Vignesh, Kafi started with a simple mission: to bring high-quality, freshly brewed coffee to the community. Inspired by South India's rich coffee heritage, Vignesh partnered with local farmers in Karnataka and Andhra Pradesh to source the finest beans. Roasted in-house, our coffee reflects the dedication and passion that go into every cup.\n\nDelivery & Locations Served\nEnjoy our coffee in the cozy café or have it delivered straight to your home or office. We proudly serve Tadipatri and nearby towns, ensuring you never miss your favorite brew.\n\nOur Menu\nFrom bold espresso blends to refreshing cold brews, artisanal teas, and fresh-baked goods, Kafi offers something for everyone. We also provide plant-based milk options and gluten-free snac

In [ ]:
# import faiss
# import numpy as np
# import json
# from sentence_transformers import SentenceTransformer

# # Load embedding model
# model = SentenceTransformer("all-MiniLM-L6-v2")  

# # Generate embeddings
# embeddings = model.encode(text_data, convert_to_numpy=True)

# # Normalize embeddings for IndexFlatIP
# embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# # Create FAISS index with Inner Product similarity
# dimension = embeddings.shape[1]
# index = faiss.IndexFlatIP(dimension)
# index.add(embeddings)

# # Store text mapping for retrieval
# id_to_text = {i: text for i, text in enumerate(text_data)}


c:\Users\vigne\OneDrive\Desktop\Viggu\sem8\mini_proj\cb_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\vigne\OneDrive\Desktop\Viggu\sem8\mini_proj\cb_venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vigne\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer M

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load the numbert model (which can handle numbers better)
model = SentenceTransformer("numerical-embedding/numbert-base")  

# Generate embeddings
embeddings = model.encode(text_data, convert_to_numpy=True)

# Normalize embeddings for IndexFlatIP
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# Create FAISS index with Inner Product similarity
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

# Store text mapping for retrieval
id_to_text = {i: text for i, text in enumerate(text_data)}


In [1]:
# Save index
faiss.write_index(index, r"cs_embeddings\number_items_faiss.index")

# Save text mappings
with open(r"cs_embeddings\number_text_mappings.json", "w", encoding="utf-8") as f:
    json.dump(id_to_text, f, indent=4)

print("Embeddings stored successfully!")

NameError: name 'faiss' is not defined

In [1]:
import faiss
import json
from sentence_transformers import SentenceTransformer
import numpy as np

# Load FAISS index
index = faiss.read_index(r"cs_embeddings/items_faiss.index")

# Load text mappings
with open(r"cs_embeddings/text_mappings.json", "r", encoding="utf-8") as f:
    id_to_text = json.load(f)

# Load the same embedding model used for indexing
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Function to search for the most similar document
def search_similar_document(query, top_k=1):
    # Convert query to embedding
    query_embedding = model.encode(query, normalize_embeddings=True)  # Normalize for inner product search
    query_embedding = np.array([query_embedding]).astype("float32")  # Convert to FAISS-compatible format

    # Search FAISS index
    distances, indices = index.search(query_embedding, top_k)

    # Retrieve the most relevant document
    results = [id_to_text[str(idx)] for idx in indices[0] if str(idx) in id_to_text]
    
    return results, distances[0]



c:\Users\vigne\OneDrive\Desktop\Viggu\sem8\mini_proj\cb_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Example Query
query_text = "i need brazilian coffee"

# Retrieve the most relevant document
retrieved_docs, scores = search_similar_document(query_text, top_k=3)

# Print results
for i, (doc, score) in enumerate(zip(retrieved_docs, scores)):
    print(f"Top {i+1} Match (Score: {score}):\n{doc}\n")

Top 1 Match (Score: 0.7275266647338867):
name: Brazilian, category: Coffee, description: A bold and smooth Brazilian coffee with rich chocolate and nutty undertones., ingredients: Brazilian Coffee Beans, Water, price: 4.0, rating: 4.8, serving_size_ml: 250, nutritional_info_per_100ml: calories: 5, sugar_g: 0, protein_g: 0.2, fat_g: 0, carbohydrates_g: 1, dietary_info: Vegetarian, Vegan

Top 2 Match (Score: 0.6938624382019043):
name: Ouro Brasileiro shot, category: Coffee, description: A rich and strong Brazilian espresso shot., ingredients: Brazilian Coffee Beans, Water, price: 3.25, rating: 4.7, serving_size_ml: 30, nutritional_info_per_100ml: calories: 1, sugar_g: 0, protein_g: 0.1, fat_g: 0, carbohydrates_g: 0.3, dietary_info: Vegetarian, Vegan

Top 3 Match (Score: 0.5238121151924133):
name: Cappuccino, category: Coffee, description: A rich and creamy coffee made with espresso, steamed milk, and a frothy milk cap., ingredients: Espresso, Steamed Milk, Milk Foam, price: 4.5, rating: 

In [49]:
retrieved_docs

["Welcome to Kafi, your neighborhood coffee shop in the heart of Tadipatri, Andhra Pradesh. At Kafi, we believe coffee is more than just a drink—it’s an experience, a moment of joy, and a way to connect with others.\n\nOur Story\nFounded in 2015 by Vignesh, Kafi started with a simple mission: to bring high-quality, freshly brewed coffee to the community. Inspired by South India's rich coffee heritage, Vignesh partnered with local farmers in Karnataka and Andhra Pradesh to source the finest beans. Roasted in-house, our coffee reflects the dedication and passion that go into every cup.\n\nDelivery & Locations Served\nEnjoy our coffee in the cozy café or have it delivered straight to your home or office. We proudly serve Tadipatri and nearby towns, ensuring you never miss your favorite brew.\n\nOur Menu\nFrom bold espresso blends to refreshing cold brews, artisanal teas, and fresh-baked goods, Kafi offers something for everyone. We also provide plant-based milk options and gluten-free sna

In [48]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")




In [49]:
# Function to get the cosine similarity score between a query and attribute embeddings
def get_similarity(query, attributes):
    query_embedding = model.encode(query)  # Get query embedding
    similarities = {}

    # Compute cosine similarity between the query and each attribute
    for attr in attributes:
        attr_embedding = model.encode(attr)  # Generate embedding for the attribute name
        similarity = cosine_similarity([query_embedding], [attr_embedding])[0][0]
        similarities[attr] = similarity
    
    return similarities



In [110]:

# List of attributes
attributes = [
    "name", 
    "category", 
    "description", 
    "ingredients", 
    "110", 
    "10",
    "rating", 
    "serving_size_ml", 
    "nutritional_info", 
    "dietary_info"
]

# Example query
query = "25"
similarities = get_similarity(query, attributes)

# Convert the similarities to a list of tuples [(attribute, similarity_score)]
similarity_list = [(attr, float(score)) for attr, score in similarities.items()]

# Print the result (list of attribute vs similarity score)
print(similarity_list[4:6])



[('110', 0.5413203239440918), ('10', 0.5278448462486267)]


In [116]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer and model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # Optional: set to eval mode for inference


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


In [117]:
def get_token_embeddings(sentence):
    # Tokenize the sentence (no offset_mapping)
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # Extract token embeddings from the model output
    token_embeddings = outputs.last_hidden_state.squeeze(0)  # shape: [seq_len, hidden_dim]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])  # Decode token IDs to tokens
    return tokens, token_embeddings



In [118]:
def extract_token_embedding(tokens, embeddings, target_token):
    for idx, token in enumerate(tokens):
        if token == target_token:
            return embeddings[idx], idx
    return None, -1



In [119]:
from torch.nn.functional import cosine_similarity

def compare_with_attributes(target_embedding, attributes):
    attr_embeddings = []
    for attr in attributes:
        # Tokenize and get the embedding for each attribute
        inputs = tokenizer(attr, return_tensors="pt")
        with torch.no_grad():
            output = model(**inputs)
        # Average the token embeddings to get sentence-level embedding
        attr_embed = output.last_hidden_state.mean(dim=1).squeeze(0)
        # Calculate cosine similarity between target and attribute embeddings
        similarity_score = cosine_similarity(target_embedding.unsqueeze(0), attr_embed.unsqueeze(0)).item()
        attr_embeddings.append((attr, similarity_score))
    return attr_embeddings


In [125]:
# Example user query
query = "I want coffee with rating above four and price below five"

# Get token embeddings for the query
tokens, token_embeddings = get_token_embeddings(query)

# Extract the contextual embedding for the word "four"
four_embedding, index = extract_token_embedding(tokens, token_embeddings, "five")

if four_embedding is not None:
    # Define attributes to compare with ("price" and "rating")
    attributes = ["price", "rating"]
    
    # Compare "four" embedding with the attributes and get cosine similarities
    similarities = compare_with_attributes(four_embedding, attributes)
    
    print("Cosine similarities with 'four':")
    for attr, score in similarities:
        print(f"{attr}: {score:.4f}")
else:
    print("'four' not found in query.")


Cosine similarities with 'four':
price: 0.1356
rating: 0.3334


In [128]:
# Function to get token embeddings
def get_token_embeddings(sentence):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    token_embeddings = outputs.last_hidden_state.squeeze(0)  # shape: [seq_len, hidden_dim]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])  # Decode token IDs to tokens
    return tokens, token_embeddings

# Function to extract token embedding for a specific token
def extract_token_embedding(tokens, embeddings, target_token):
    for idx, token in enumerate(tokens):
        if token == target_token:
            return embeddings[idx], idx
    return None, -1

# Function to compare "five" and "four" embeddings with attributes (price, rating)
def compare_with_attributes(target_embeddings, attributes):
    attr_embeddings = []
    for attr in attributes:
        # Tokenize and get the embedding for each attribute
        inputs = tokenizer(attr, return_tensors="pt")
        with torch.no_grad():
            output = model(**inputs)
        attr_embed = output.last_hidden_state.mean(dim=1).squeeze(0)
        
        # Calculate cosine similarity for each target embedding
        similarities = []
        for target_embedding in target_embeddings:
            similarity_score = cosine_similarity(target_embedding.unsqueeze(0), attr_embed.unsqueeze(0)).item()
            similarities.append(similarity_score)
        
        # Debug: Print out the similarity scores for each attribute
        print(f"Similarity scores for attribute '{attr}': {similarities}")
        
        attr_embeddings.append((attr, similarities))
    return attr_embeddings

# Full example usage
query = "I want coffee with rating above five and price below five"

# Get token embeddings for the query
tokens, token_embeddings = get_token_embeddings(query)

# Extract the contextual embeddings for the words "five" and "four"
target_tokens = ["five", "four"]
target_embeddings = []

for target_token in target_tokens:
    embedding, index = extract_token_embedding(tokens, token_embeddings, target_token)
    if embedding is not None:
        target_embeddings.append(embedding)
    else:
        print(f"'{target_token}' not found in query.")

# Debug: Check the embeddings for "five" and "four"
print(f"Extracted embeddings for 'five' and 'four': {target_embeddings}")

# Define attributes to compare with ("price" and "rating")
attributes = ["price", "rating"]

# Compare "five" and "four" embeddings with the attributes and get cosine similarities
similarities = compare_with_attributes(target_embeddings, attributes)

# Print cosine similarities for both "five" and "four"
print("Cosine similarities with 'five' and 'four':")
for attr, scores in similarities:
    # Ensure to print both values for "five" and "four"
    print(f"{attr}: 'five' = {scores[0]:.4f}, 'four' = {scores[1]:.4f}")

'four' not found in query.
Extracted embeddings for 'five' and 'four': [tensor([-4.7889e-02, -3.0008e-01,  1.7471e-01,  2.1562e-02, -2.2907e-02,
         3.4848e-01, -1.6660e-01, -2.4572e-02, -1.1999e-02,  1.1946e-01,
        -2.0982e-01, -4.6066e-01, -1.3450e-01,  5.2713e-03, -4.0552e-01,
        -1.8281e-02,  6.7243e-02, -1.1434e-01, -7.0768e-01, -5.5334e-01,
         4.4088e-01, -2.2989e-01,  2.0774e-01,  1.5442e-01,  1.8940e-01,
        -6.5908e-02, -2.3420e-01, -1.4807e-01, -3.9672e-01,  3.8107e-02,
        -4.1034e-01,  3.8275e-01,  1.0154e-01, -1.3324e-02, -3.7211e-01,
        -1.5101e-01, -2.5414e-02, -2.1730e-01,  3.1821e-02,  4.7917e-01,
        -5.6688e-02, -1.9801e-01, -5.3139e-01, -1.1373e-01,  1.7215e-02,
        -1.2192e-01, -4.8581e-01,  2.9078e-01,  2.7656e-01, -2.3003e-01,
        -3.3020e-01,  1.0877e-01, -1.9239e-01, -1.9510e-01, -4.2419e-02,
        -9.4135e-01, -4.2333e-01, -1.9552e-01,  2.2261e-02, -1.2866e-01,
         4.7897e-02, -8.8006e-02, -1.7484e-01, -2.48

IndexError: list index out of range